# 6. Hafta — Rastgele ölüm içeren Game of Life

**Proje:** Hücresel Otomatlarda Beliren Karmaşıklık  
**Süre:** Yaklaşık 10 saat  
**Bu haftanın araştırma sorusu:** *Yerel kurala küçük bir rastgele ölüm olasılığı eklemek, uzun zamandaki etkinliği ve sönme olasılığını nasıl değiştirir?*

Bu defter bir **öğrenci çalışma şablonudur**. `TODO` satırlarını tamamla, hemen altlarındaki testleri çalıştır, deney hücrelerini aç ve boş bırakılan yorumları kendi gözlemlerinle doldur. Bir yapay zekâ aracından yardım alırsan hangi bölümde ve ne için kullandığını sonuç kısmında belirt.


## Şimdiye kadar: yerel kuraldan istatistiksel deneye

- **1. hafta:** Basit bir yerel kuralın zaman içinde büyük ölçekli desenler üretebildiğini gördün.
- **2. hafta:** Kuralları görüntü, yoğunluk ve etkinlik gibi gözlenebilirlerle karşılaştırdın; tek bir ölçümün tüm davranışı anlatmadığını fark ettin.
- **3. hafta:** Başlangıç koşullarındaki değişimlerin sonuçları nasıl etkileyebildiğini inceledin.
- **4. hafta:** İki boyutlu, deterministik B3/S23 Game of Life motorunu kurdun; sabit yapıları, salınıcıları ve hareket ediyormuş gibi görünen desenleri tanıdın.
- **5. hafta:** Aynı kuralı birçok bağımsız rastgele başlangıçla çalıştırdın. Başlangıç canlılık olasılığı $p$ ile geç-zaman yoğunluğu/etkinliği arasındaki ilişkiyi, ortalamayı ve deneyler arası saçılmayı ölçtün.

Geçen hafta **kural sabitti, başlangıç rastgeleydi**. Bu hafta başlangıç protokolünü sabit tutup kuralın içine kontrol edilebilir bir rastgelelik ekleyeceğiz. Böylece *başlangıç belirsizliği* ile *dinamikteki rastgeleliği* birbirinden ayırmayı öğreneceksin.

**7. haftaya köprü:** Bu haftaki taramadan bir soru seçeceksin: örneğin ızgara boyutunun, gözlem süresinin veya başlangıç yoğunluğunun sonuçlara etkisi. 7. haftada bu soruyu daha bağımsız bir sayısal araştırmaya dönüştüreceksin. 8. haftada ise yöntemini, sınırlamalarını ve en güçlü şekillerini bir araştırma raporunda birleştireceksin.


## 1. Model: iki aşamalı bir güncelleme

Önce her hücreye standart B3/S23 Life kuralını **eşzamanlı** uygula. Ortaya çıkan ızgaradaki her canlı hücre, bundan **sonra**, birbirinden bağımsız olarak $\mu$ olasılıkla ölsün. Ölü hücreler bu ikinci aşamada doğmaz.

$$G_{t+1}=D_{\mu}\!\left(F_{\mathrm{Life}}(G_t)\right).$$

Burada $F_{\mathrm{Life}}$ deterministik Life güncellemesi; $D_{\mu}$ ise canlı hücreleri rastgele silen aşamadır. $\mu$ bir *olasılık* olduğu için $0\leq\mu\leq1$.

- $\mu=0$: Standart, deterministik Game of Life.
- $\mu=1$: Life güncellemesinden sonra bütün canlı hücreler silinir; ilk güncellemenin sonunda ızgara boştur.
- Tamamen boş bir ızgara bu modelde **soğurucu durumdur**: kendiliğinden canlı hücre oluşmaz.

**Önemli:** Rastgele ölümü Life kuralından *önce* uygulamak farklı bir modeldir. Bu defterde sıra her zaman **Life → rastgele silme** olacaktır. Izgara kenarlarında 4. haftadaki periyodik sınır koşullarını kullanıyoruz.

<details><summary>Kontrol sorusu: $\mu=0$ ve $\mu=1$ için ne olmalı? Yanıtı aç.</summary>

$\mu=0$ için standart Life ile hücre hücre aynı geçmiş elde edilmelidir. $\mu=1$ için başlangıç $t=0$ karesi korunur, fakat $t=1$ ve sonraki kareler tamamen boştur.
</details>


### Deneyden önce tahmin et

1. $\mu$ arttıkça geç-zaman yoğunluğu ve etkinliği nasıl değişebilir? Monotonluk bekliyor musun?
2. $\mu$ çok küçük olsa bile sabit bir blok sonsuza dek sabit kalır mı?
3. Bir çalışmada etkinlik sıfır ve yoğunluk pozitif olabilir mi? $\mu>0$ için bunun yorumu nedir?
4. $M$ kez tekrar etmemizin nedeni nedir? Rastgele başlangıç ile rastgele dinamiğin rollerini ayırabilir misin?

**Tahminlerim (deneyi çalıştırmadan önce doldur):**

> ...


In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 120
plt.rcParams["image.cmap"] = "binary"


def check(label, test_function):
    """TODO tamamlanmasa bile sonraki hücrelere geçilebilsin."""
    try:
        test_function()
    except (AssertionError, NotImplementedError, ValueError, TypeError) as error:
        message = str(error) or "sonuç beklenen değerle eşleşmedi"
        print(f"○ {label}: henüz geçmedi — {message}")
    else:
        print(f"✓ {label}: geçti")


print("Kurulum tamamlandı.")


## 2. Geçen haftadan sağlam bir deney motoru

Aşağıdaki fonksiyonlar **tam verilmiştir**; bu haftanın yeni alıştırması değildir. `simulate_life` fonksiyonunda `steps`, başlangıç karesi dahil toplam kare sayısıdır. Buradaki `activity_time_series` özellikle `uint8` taşma hatasından korunmak için karşılaştırma kullanır.


In [ ]:
def count_neighbors(grid):
    """Periyodik sınırlarda sekiz canlı komşuyu say."""
    grid = np.asarray(grid, dtype=np.uint8)
    neighbors = np.zeros_like(grid, dtype=np.uint8)
    for di in (-1, 0, 1):
        for dj in (-1, 0, 1):
            if di == 0 and dj == 0:
                continue
            neighbors += np.roll(np.roll(grid, di, axis=0), dj, axis=1)
    return neighbors


def update_life(grid):
    """B3/S23 kuralını bir kez uygula."""
    grid = np.asarray(grid, dtype=np.uint8)
    neighbors = count_neighbors(grid)
    birth = (grid == 0) & (neighbors == 3)
    survive = (grid == 1) & ((neighbors == 2) | (neighbors == 3))
    return (birth | survive).astype(np.uint8)


def simulate_life(initial_grid, steps):
    initial_grid = np.asarray(initial_grid, dtype=np.uint8)
    history = np.zeros((steps, *initial_grid.shape), dtype=np.uint8)
    history[0] = initial_grid
    for t in range(1, steps):
        history[t] = update_life(history[t - 1])
    return history


def random_grid(size, density, rng):
    return (rng.random((size, size)) < density).astype(np.uint8)


def density_time_series(history):
    return np.asarray(history).mean(axis=(1, 2))


def activity_time_series(history):
    history = np.asarray(history)
    return (history[1:] != history[:-1]).mean(axis=(1, 2))


def late_time_average(values, window):
    values = np.asarray(values, dtype=float)
    if window < 1 or window > len(values):
        raise ValueError("window veri uzunluğu içinde olmalı")
    return float(values[-window:].mean())


assert np.array_equal(update_life(np.zeros((5, 5), dtype=np.uint8)), np.zeros((5, 5), dtype=np.uint8))
print("Life motoru ve gözlenebilirler hazır.")


## 3. Rastgele silme aşaması

### Alıştırma 1 — `apply_random_death`

Bir 0/1 ızgarasındaki **canlı** hücreleri $\mu$ olasılıkla sil. Bunun için `rng.random(grid.shape)` ile her konum için bağımsız bir sayı çekebilirsin. Giriş ızgarasını değiştirme; yeni bir `uint8` ızgara döndür.

**İpucu:** Bir hücre ancak eskiden canlıysa *ve* çekilen sayı $\mu$'dan küçük değilse canlı kalır. Uç durumlar için `mu=0` ve `mu=1` testlerini düşün.


In [ ]:
def apply_random_death(grid, mu, rng):
    grid = np.asarray(grid, dtype=np.uint8)
    if not 0 <= mu <= 1:
        raise ValueError("mu 0 ile 1 arasında olmalı")
    # TODO: her konum için rng.random(grid.shape) ile sayı çek.
    # TODO: yalnızca hayatta kalan hücrelerden yeni uint8 ızgara üret.
    raise NotImplementedError("Rastgele silme aşamasını tamamla")


In [ ]:
def test_apply_random_death():
    grid = np.array([[1, 0, 1], [1, 1, 0]], dtype=np.uint8)
    original = grid.copy()
    assert np.array_equal(apply_random_death(grid, 0.0, np.random.default_rng(3)), grid)
    assert np.array_equal(apply_random_death(grid, 1.0, np.random.default_rng(3)), np.zeros_like(grid))
    seed = 84
    expected_draws = np.random.default_rng(seed).random(grid.shape)
    expected = ((grid == 1) & (expected_draws >= 0.4)).astype(np.uint8)
    actual = apply_random_death(grid, 0.4, np.random.default_rng(seed))
    assert np.array_equal(actual, expected), "eşik veya rastgele sayı kullanımı farklı"
    assert np.array_equal(grid, original), "giriş ızgarası değiştirilmemeli"
    assert actual.dtype == np.uint8
    try:
        apply_random_death(grid, 1.2, np.random.default_rng(seed))
    except ValueError:
        pass
    else:
        raise AssertionError("mu aralık dışında ise ValueError bekleniyor")


check("Alıştırma 1 — rastgele silme", test_apply_random_death)


<details><summary>Soru: Neden her hücre için yeni rastgele sayı çekiyoruz? Yanıtı aç.</summary>

Tek bir sayı çekip bütün ızgaraya uygularsak bütün hücreler birlikte silinir veya birlikte kalır. Bu, bağımsız hücre ölümleri modeli değil, farklı bir ortak-gürültü modelidir.
</details>

**Kendi açıklamam:** ...


## 4. Tek adımlık ve çok adımlık gürültülü Life

### Alıştırma 2 — `update_noisy_life`

Sırayı karıştırma: önce `update_life`, sonra `apply_random_death`. `mu=0` için sonuç standart Life ile tam aynı olmalı.


In [ ]:
def update_noisy_life(grid, mu, rng):
    # TODO: önce deterministik Life güncellemesini hesapla.
    # TODO: onun sonucuna rastgele silmeyi uygula ve döndür.
    raise NotImplementedError("İki aşamalı güncellemeyi tamamla")


In [ ]:
def test_update_noisy_life():
    grid = np.zeros((7, 7), dtype=np.uint8)
    grid[3, 2:5] = 1  # yatay blinker
    expected_life = update_life(grid)
    assert np.array_equal(update_noisy_life(grid, 0.0, np.random.default_rng(1)), expected_life)
    assert not update_noisy_life(grid, 1.0, np.random.default_rng(1)).any()
    assert np.array_equal(grid[3, 2:5], np.ones(3, dtype=np.uint8)), "giriş değişmemeli"


check("Alıştırma 2 — gürültülü tek adım", test_update_noisy_life)


### Alıştırma 3 — `simulate_noisy_life`

`history[0]` başlangıç ızgarası olsun; diğer kareleri bir önceki kareden üret. Aynı `rng` nesnesini her adımda kullan. Fonksiyon `steps` kare döndürür, `steps` güncelleme yapmaz.


In [ ]:
def simulate_noisy_life(initial_grid, steps, mu, rng):
    if steps < 2:
        raise ValueError("steps en az 2 olmalı")
    initial_grid = np.asarray(initial_grid, dtype=np.uint8)
    history = np.zeros((steps, *initial_grid.shape), dtype=np.uint8)
    history[0] = initial_grid
    for t in range(1, steps):
        # TODO: history[t] değerini önceki kareden üret.
        raise NotImplementedError("Zaman döngüsünü tamamla")
    return history


In [ ]:
def test_simulate_noisy_life():
    initial = np.zeros((7, 7), dtype=np.uint8)
    initial[3, 2:5] = 1
    original = initial.copy()
    h0 = simulate_noisy_life(initial, 5, 0.0, np.random.default_rng(12))
    assert h0.shape == (5, 7, 7)
    assert np.array_equal(h0, simulate_life(initial, 5)), "mu=0 standart Life ile aynı olmalı"
    h1 = simulate_noisy_life(initial, 5, 1.0, np.random.default_rng(12))
    assert np.array_equal(h1[0], initial)
    assert not h1[1:].any(), "mu=1 ilk güncellemeden sonra söner"
    assert np.array_equal(initial, original), "başlangıç ızgarası değişmemeli"
    h_a = simulate_noisy_life(initial, 5, 0.2, np.random.default_rng(99))
    h_b = simulate_noisy_life(initial, 5, 0.2, np.random.default_rng(99))
    assert np.array_equal(h_a, h_b), "aynı tohum tekrarlanabilir olmalı"


check("Alıştırma 3 — çok adımlı benzetim", test_simulate_noisy_life)


## 5. Yeni bir gözlenebilir: sönme

Geç-zaman yoğunluğu ve etkinliğine ek olarak ilk sönme zamanını ölçeceğiz:

$$\tau_{\mathrm{ext}}=\min\{t: \rho(t)=0\}.$$

Sistem gözlem süresince hiç sönmezse `None` döndür. Bu, **sonsuz süre yaşayacağı** anlamına gelmez; yalnızca kullandığımız $T$ süresinde sönmemiştir. Başlangıç ızgarası boşsa sönme zamanı `0`'dır.

### Alıştırma 4 — `first_extinction_time`

**İpucu:** Her zaman karesi için `frame.any()` canlı hücre olup olmadığını söyler. `enumerate(history)` ile zaman indeksini alabilirsin.


In [ ]:
def first_extinction_time(history):
    # TODO: ilk tamamen boş karenin indeksini döndür; yoksa None.
    raise NotImplementedError("İlk sönme zamanını bul")


In [ ]:
def test_first_extinction_time():
    alive = np.array([[1, 0], [0, 0]], dtype=np.uint8)
    empty = np.zeros((2, 2), dtype=np.uint8)
    assert first_extinction_time(np.array([alive, alive, empty, empty])) == 2
    assert first_extinction_time(np.array([empty, empty])) == 0
    assert first_extinction_time(np.array([alive, alive])) is None


check("Alıştırma 4 — sönme zamanı", test_first_extinction_time)


<details><summary>Soru: Sönmüş bir durum tekrar canlanabilir mi? Yanıtı aç.</summary>

Hayır. Boş ızgarada hiçbir hücrenin üç canlı komşusu yoktur. Life aşaması boşluğu korur; rastgele silme aşaması da canlı hücre yaratmaz. Bu yüzden boş durum soğurucudur.
</details>

<details><summary>İleri düşünme: Sonlu bir ızgarada $\mu>0$ ise sonsuz zamanda ne bekleriz?</summary>

Life aşamasından sonra en çok $L^2$ canlı hücre vardır. Bunların hepsinin aynı güncellemede silinmesi olasılığı en az $\mu^{L^2}>0$'dır. Dolayısıyla sonlu bir ızgarada sonsuza kadar tekrar tekrar şans verildiğinde sönme olasılığı 1'e gider. Bu, sonlu $T$ boyunca gözlediğimiz canlı/etkin rejimin anlamsız olduğu demek değildir; **hangi zaman ölçeğinde** baktığımızı açıkça söylememiz gerektiği demektir. Sonsuz boyut limiti ayrı ve daha ileri bir sorudur.
</details>


## 6. Aynı parametrelerde bağımsız tekrarlar

Her tekrar için yeni bir başlangıç ızgarası oluştur ve zaman boyunca yeni rastgele ölümler üret. **Aynı başlangıç yoğunluğu $p_0$, ızgara boyutu $L$ ve gözlem süresi $T$**, bütün $\mu$ değerlerinde korunmalı. Başlangıcı tek sefer üretip her tekrarda aynen kullanırsan başlangıç belirsizliğini ölçemezsin.

Çıktılar üç adet uzunluk-`runs` dizisi olsun: geç-zaman yoğunlukları, geç-zaman etkinlikleri ve sönme zamanları. Gözlem süresince sönmeyen deneyin sönme zamanı için `np.nan` kullan.

### Alıştırma 5 — `run_noisy_ensemble`

**İpucu:** Geçen haftadaki gibi yoğunluk için `late_window` son kareyi, etkinlik için son `late_window` geçişi ortala. Etkinlik dizisinin uzunluğu `steps-1` olduğundan `min(late_window, len(...))` kullanabilirsin.


In [ ]:
def run_noisy_ensemble(size, p0, steps, runs, late_window, mu, rng):
    if runs < 2 or steps < 2 or not 1 <= late_window <= steps - 1:
        raise ValueError("runs>=2, steps>=2 ve 1<=late_window<=steps-1 gerekli")
    late_rho = np.zeros(runs, dtype=float)
    late_activity = np.zeros(runs, dtype=float)
    extinction_times = np.full(runs, np.nan, dtype=float)

    for k in range(runs):
        # TODO: yeni bir random_grid üret ve simulate_noisy_life ile çalıştır.
        # TODO: geç-zaman yoğunluğunu ve etkinliğini hesapla.
        # TODO: sönme varsa zamanını extinction_times[k] içine yaz.
        raise NotImplementedError("Tekrar döngüsünü tamamla")

    return late_rho, late_activity, extinction_times


In [ ]:
def test_run_noisy_ensemble():
    rho, activity, tau = run_noisy_ensemble(
        size=6, p0=1.0, steps=4, runs=3, late_window=2,
        mu=1.0, rng=np.random.default_rng(7)
    )
    assert rho.shape == activity.shape == tau.shape == (3,)
    assert np.allclose(rho, 0) and np.allclose(activity, 0)
    assert np.array_equal(tau, np.ones(3)), "mu=1 ilk güncellemede sönmeli"
    a = run_noisy_ensemble(12, 0.35, 20, 4, 5, 0.02, np.random.default_rng(77))
    b = run_noisy_ensemble(12, 0.35, 20, 4, 5, 0.02, np.random.default_rng(77))
    for x, y in zip(a, b):
        assert np.array_equal(x, y, equal_nan=True), "aynı tohum tekrarlanabilir olmalı"
    assert np.all((a[0] >= 0) & (a[0] <= 1))
    assert np.all((a[1] >= 0) & (a[1] <= 1))


check("Alıştırma 5 — bağımsız tekrarlar", test_run_noisy_ensemble)


### Ensemble özetinin anlamı

Bir $\mu$ için $M$ tekrarın geç-zaman yoğunlukları $x_1,\ldots,x_M$ olsun. Şunları raporlayacağız:

$$\bar x=\frac{1}{M}\sum_k x_k,\qquad
s_x=\sqrt{\frac{1}{M-1}\sum_k(x_k-\bar x)^2},\qquad
\mathrm{SE}(\bar x)=\frac{s_x}{\sqrt M}.$$

$s_x$ deneyler arası saçılmayı, $\mathrm{SE}$ ise ortalamanın tahmin belirsizliğini anlatır. Sönme oranı $\widehat P_{\mathrm{ext}}(T)$, $T$ karelik gözlemde sönmüş tekrarların oranıdır; sonsuz-zaman olasılığı değildir.


In [ ]:
def summarize_ensemble(late_rho, late_activity, extinction_times):
    runs = len(late_rho)
    if not (len(late_activity) == runs == len(extinction_times)) or runs < 2:
        raise ValueError("aynı uzunlukta en az iki tekrar gerekli")
    return {
        "rho_mean": float(np.mean(late_rho)),
        "rho_sd": float(np.std(late_rho, ddof=1)),
        "rho_se": float(np.std(late_rho, ddof=1) / np.sqrt(runs)),
        "activity_mean": float(np.mean(late_activity)),
        "activity_sd": float(np.std(late_activity, ddof=1)),
        "activity_se": float(np.std(late_activity, ddof=1) / np.sqrt(runs)),
        "extinction_fraction": float(np.isfinite(extinction_times).mean()),
    }


assert summarize_ensemble([0, 1], [0, 1], [1, np.nan])["extinction_fraction"] == 0.5
print("Ensemble özeti hazır.")


## 7. Pilot: tarama aralığını akıllıca seç

Önce küçük bir tarama yap. Başlangıç için $p_0=0.35$ seçiyoruz; 5. haftada bu bölgede ölçülebilir geç-zaman etkinliği vardı. Başka $p_0$ seçersen nedenini yaz ve bütün $\mu$ değerlerinde aynı tut.

**Çalıştırmadan önce:** Alıştırma 1–5 testlerinin hepsi geçmeli. Sonra `RUN_PILOT = True` yap.


In [ ]:
RUN_PILOT = False  # Beş test geçtikten sonra True yap.

pilot_size = 40
pilot_p0 = 0.35
pilot_steps = 120
pilot_runs = 5
pilot_window = 30
pilot_mu_values = np.array([0.0, 0.005, 0.01, 0.02, 0.04, 0.06, 0.10, 0.20, 1.0])
pilot_results = {}

if RUN_PILOT:
    pilot_rng = np.random.default_rng(202606)
    start = time.perf_counter()
    for mu in pilot_mu_values:
        samples = run_noisy_ensemble(
            pilot_size, pilot_p0, pilot_steps, pilot_runs,
            pilot_window, float(mu), pilot_rng
        )
        pilot_results[float(mu)] = summarize_ensemble(*samples)
    print(f"Pilot süresi: {time.perf_counter() - start:.2f} saniye")

    fig, axes = plt.subplots(1, 3, figsize=(13, 3.8), constrained_layout=True)
    for ax, key, label in zip(
        axes,
        ("rho_mean", "activity_mean", "extinction_fraction"),
        ("Geç yoğunluk", "Geç etkinlik", "T'ye kadar sönme oranı")
    ):
        ax.plot(pilot_mu_values, [pilot_results[float(m)][key] for m in pilot_mu_values], "o-")
        ax.set(xlabel="Rastgele ölüm olasılığı μ", ylabel=label)
        ax.grid(alpha=0.25)
    plt.show()
else:
    print("Pilot henüz çalıştırılmadı. Testlerden sonra RUN_PILOT=True yap.")


### Pilot yorumu — öğrenci dolduracak

1. Pilot kaç saniye sürdü? Ana tarama için süreyi yaklaşık nasıl tahmin edersin?
2. Değişim hangi $\mu$ aralığında yoğunlaşıyor? Eşit aralıklı geniş bir tarama mı, yoksa sıfıra yakın sık noktalar mı daha yararlı?
3. $\mu=0$ ve $\mu=1$ uç noktaları beklediğin gibi mi?
4. Aynı $\mu$ değerinde farklı tekrarlar benzer mi? Pilotun $M=5$ sayısı kesin bir sonuç için yeterli mi?

**Yanıtlarım:**

> ...


## 8. Ana deney: tek kontrol parametresi olarak $\mu$

Öneri: $L=60$, $T=250$, $M=20$, $p_0=0.35$, geç-zaman penceresi 50. Sıfıra yakın daha ince, uzaklarda daha seyrek $\mu$ noktaları kullan. **Yalnızca $\mu$ değişmeli.** Pilot çok yavaşsa parametreleri makul biçimde küçült ve bunu raporla.

Bu bir **sonlu boyutlu ve sonlu süreli** sayısal taramadır. Grafikte keskin görünen bir değişim bulsan bile yalnızca bu veriye dayanarak “kritik nokta” veya “faz geçişi” deme.

**Çalıştırmadan önce:** `RUN_MAIN_EXPERIMENT = True` yap. Pilot sonuçlarını yorumladıktan sonra gerekirse `main_mu_values` dizisini değiştir ve seçimini açıkla.


In [ ]:
RUN_MAIN_EXPERIMENT = False  # Pilot başarılı olduktan sonra True yap.

main_size = 60
main_p0 = 0.35
main_steps = 250
main_runs = 20
main_window = 50
main_mu_values = np.array([
    0.0, 0.002, 0.005, 0.01, 0.015, 0.02, 0.03,
    0.04, 0.05, 0.06, 0.08, 0.10, 0.20, 1.0
])
main_results = {}

if RUN_MAIN_EXPERIMENT:
    main_rng = np.random.default_rng(20260606)
    start = time.perf_counter()
    for mu in main_mu_values:
        late_rho, late_activity, extinction_times = run_noisy_ensemble(
            main_size, main_p0, main_steps, main_runs,
            main_window, float(mu), main_rng
        )
        main_results[float(mu)] = {
            **summarize_ensemble(late_rho, late_activity, extinction_times),
            "rho_samples": late_rho,
            "activity_samples": late_activity,
            "extinction_times": extinction_times,
        }
    print(f"Ana deney süresi: {time.perf_counter() - start:.2f} saniye")
else:
    print("Ana deney henüz çalıştırılmadı. Hazır olunca RUN_MAIN_EXPERIMENT=True yap.")


### Ana sonuç şekilleri

İlk iki paneldeki hata çubukları **ortalamanın standart hatasıdır (SE)**. Bunlar deneyler arası standart sapma değildir. Son panel, sönme oranını verir. Bu oran özellikle gözlem süresi $T$'ye bağlıdır.


In [ ]:
if main_results:
    rho_mean = [main_results[float(m)]["rho_mean"] for m in main_mu_values]
    rho_se = [main_results[float(m)]["rho_se"] for m in main_mu_values]
    activity_mean = [main_results[float(m)]["activity_mean"] for m in main_mu_values]
    activity_se = [main_results[float(m)]["activity_se"] for m in main_mu_values]
    extinction_fraction = [main_results[float(m)]["extinction_fraction"] for m in main_mu_values]

    fig, axes = plt.subplots(1, 3, figsize=(14, 4), constrained_layout=True)
    axes[0].errorbar(main_mu_values, rho_mean, yerr=rho_se, fmt="o-", capsize=3)
    axes[0].set(xlabel="μ", ylabel="Ortalama geç yoğunluk ± SE")
    axes[1].errorbar(main_mu_values, activity_mean, yerr=activity_se, fmt="o-", capsize=3, color="tab:red")
    axes[1].set(xlabel="μ", ylabel="Ortalama geç etkinlik ± SE")
    axes[2].plot(main_mu_values, extinction_fraction, "o-", color="tab:purple")
    axes[2].set(xlabel="μ", ylabel="T'ye kadar sönme oranı", ylim=(-0.05, 1.05))
    for ax in axes:
        ax.grid(alpha=0.25)
    plt.show()
else:
    print("Grafikler için önce ana deneyi çalıştır.")


### Sonuç tablosu ve sayısal kontrol

Bu hücre ana taramanın tüm noktalarını özetler. Tablodan en az beş $\mu$ değeri seçip aşağıdaki yorum bölümünde sayılarla karşılaştır.


In [ ]:
if main_results:
    print("mu     rho_ortalama  rho_SE    etkinlik_ortalama  etkinlik_SE  sonme_orani")
    for mu in main_mu_values:
        r = main_results[float(mu)]
        print(f"{mu:0.3f}  {r['rho_mean']:0.5f}       {r['rho_se']:0.5f}   "
              f"{r['activity_mean']:0.5f}             {r['activity_se']:0.5f}       "
              f"{r['extinction_fraction']:0.2f}")
else:
    print("Tablo için önce ana deneyi çalıştır.")


### Ana sonuçları yorumla — öğrenci dolduracak

1. $\mu=0$ sonucunu 5. haftadaki **aynı $p_0$ için**, fakat farklı $T$ nedeniyle yalnızca yaklaşık, sonuçla karşılaştır. Beklenen farklar neler?
2. Hangi $\mu$ aralığında geç-zaman yoğunluğu ve etkinliği belirgin biçimde azalıyor? Birkaç sayısal değer ver.
3. Sönme oranı nerede artıyor? $M=20$ için bu oran yalnızca $1/20$ basamaklarıyla değişebilir; bunu yorumlarken dikkate al.
4. En yüksek yoğunluk/etkinlik noktası ile sönmenin yaygınlaştığı bölge aynı şey midir?
5. Hata çubukları neyi gösteriyor? Aynı grafiği standart sapma ile çizseydin hangi farklı soruyu yanıtlamış olurdun?
6. $T=250$ sonunda sönmemiş bir deneyin *hiç sönmeyeceği* sonucunu çıkarabilir misin?

**Yanıtlarım (ölçüm ile yorum cümlesini ayır):**

> ...


## 9. Aynı başlangıç, farklı rastgele dinamikler

Bu küçük kontrol deneyinde **tek bir başlangıç ızgarasını** sabit tutup aynı $\mu$ ile farklı rastgele tohumlar kullan. Böylece 5. haftadaki farklı başlangıçlar etkisinden ayrı olarak dinamikteki rastgeleliğin etkisini görürsün. İlk hücrede uygun $\mu$ seçimini ve nedenini yaz.

**İpucu:** `rng` nesnelerini hem başlangıç hem evrim için karıştırma; başlangıcı bir kez oluştur, üç evrim için üç ayrı üreteç kullan.


In [ ]:
RUN_CONTROL_EXPERIMENT = False  # Ana tarama tamamlanınca True yap.
control_mu = 0.02  # Ana grafiklerine bakıp istersen değiştir.

if RUN_CONTROL_EXPERIMENT:
    fixed_initial = random_grid(main_size, main_p0, np.random.default_rng(6400))
    control_histories = [
        simulate_noisy_life(fixed_initial, main_steps, control_mu, np.random.default_rng(seed))
        for seed in (6401, 6402, 6403)
    ]
    assert all(np.array_equal(h[0], fixed_initial) for h in control_histories)

    fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), constrained_layout=True)
    for i, history in enumerate(control_histories, start=1):
        axes[0].plot(density_time_series(history), label=f"Dinamik tohumu {i}")
        axes[1].plot(activity_time_series(history), label=f"Dinamik tohumu {i}")
    axes[0].set(xlabel="Zaman", ylabel="Yoğunluk")
    axes[1].set(xlabel="Zaman", ylabel="Etkinlik")
    for ax in axes:
        ax.grid(alpha=0.25)
        ax.legend()
    plt.show()

    print("İlk sönme zamanları:", [first_extinction_time(h) for h in control_histories])
else:
    print("Kontrol deneyini açmak için RUN_CONTROL_EXPERIMENT=True yap.")


### Kontrol deneyini yorumla — öğrenci dolduracak

Üç eğri aynı mı? İlk ayrışma ne zaman görülüyor? Farkları başlangıç ızgarasına mı, yoksa dinamikteki rastgele çekilişlere mi bağlayabilirsin? Bu deneyin 5. haftadaki ensemble deneyinden tam olarak farkı nedir?

**Yanıtım:**

> ...


## 10. Küçük bir sağlamlık kontrolü

Ana grafikten değişimin belirgin olduğu **tek bir $\mu$** seç. Aynı $L,p_0,M$ ile $T=150$ ve $T=350$ için sönme oranlarını karşılaştır. İstersen sönme oranı için basit binom standart hatasını $\sqrt{\hat q(1-\hat q)/M}$ olarak hesapla; $\hat q=0$ veya $1$ iken bu yaklaşık formülün fazla iyimser olabileceğini unutma.

Bu bölümde kodu sen yaz; aşağıdaki boş hücreyi kullan. Hesaplama çok uzunsa yalnızca $M=10$ ile başla ve sınırlamayı belirt.


In [ ]:
# TODO (araştırma görevi): Bir mu seçip T=150 ve T=350 için sönme oranını karşılaştır.
# Sonuçları kısa bir tablo veya şekille göster.


**Sağlamlık kontrolü yorumum:**

> ...


## 11. Sonuç, sınırlamalar ve 7. haftaya araştırma sorusu

Kısa bir bilimsel sonuç yaz. Şu noktaların hepsine değin:

1. **Model:** Rastgele ölüm *ne zaman* uygulandı? $\mu=0$ neyi temsil ediyor?
2. **Yöntem:** Hangi $L,T,M,p_0$ ve $\mu$ değerlerini kullandın? Kaç bağımsız tekrar yaptın?
3. **Bulgular:** En az üç sayısal sonuç ve bir şekil referansı ver. “Gördüm/ölçtüm” ile “bunun nedeni olabilir” cümlelerini ayır.
4. **Sınırlamalar:** Sonlu ızgara, sonlu süre, tekrar sayısı, seçilen $p_0$ ve sönmemiş deneylerin sansürlenmesi.
5. **7. hafta:** Bu haftadan doğan *tek, sınanabilir* araştırma sorusu seç. Hangi parametreyi değiştirecek, hangi gözlenebiliri ölçecek, neyi sabit tutacaksın?

**Sonucum:**

> ...

**7. haftada sınamak istediğim soru ve planım:**

> ...

**AI kullanımı (kullandıysan ne için; kullanmadıysan belirt):**

> ...

### Teslim kontrol listesi

- [ ] Beş alıştırma testi geçti.
- [ ] Pilot, ana tarama ve kontrol deneyi çalıştırıldı; sonuçlar defterde görünüyor.
- [ ] En az bir sağlamlık kontrolü yapıldı.
- [ ] Şekil ve tablolar açıklanıp belirsizlik türleri doğru adlandırıldı.
- [ ] Sonuç ve 7. hafta araştırma sorusu kendi cümlelerimle yazıldı.
- [ ] Defteri **Restart & Run All** ile yeniden çalıştırdım.
